In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go

linked_points = [
    {"sample": "P0", "time": 0, "derived": 0.05, "source": [0.04, 0.05, 0.06, 0.05]},
    {"sample": "P1", "time": 2, "derived": 0.14, "source": [0.12, 0.15, 0.13, 0.16]},
    {"sample": "P2", "time": 4, "derived": 0.29, "source": [0.27, 0.30, 0.28, 0.31]},
    {"sample": "P3", "time": 6, "derived": 0.43, "source": [0.40, 0.45, 0.43, 0.44]},
]

fig_a = go.FigureWidget(
    data=[
        go.Scatter(
            x=[p["time"] for p in linked_points],
            y=[p["derived"] for p in linked_points],
            mode="lines+markers",
            name="derived data",
        )
    ],
    layout=go.Layout(
        title="Plot A: derived values",
        xaxis_title="Reaction time (min)",
        yaxis_title="Concentration (mM)",
        hovermode="closest",
    ),
)

fig_b = go.FigureWidget(
    data=[
        go.Scatter(mode="markers", name="source replicates"),
        go.Scatter(mode="lines", name="mean used in Plot A"),
    ],
    layout=go.Layout(
        title="Plot B: source data",
        xaxis_title="Replicate",
        yaxis_title="Estimated concentration (mM)",
    ),
)


def update_bottom(i):
    src = linked_points[i]["source"]
    mean = linked_points[i]["derived"]
    x = list(range(1, len(src) + 1))

    with fig_b.batch_update():
        fig_b.data[0].x = x
        fig_b.data[0].y = src
        fig_b.data[1].x = x
        fig_b.data[1].y = [mean] * len(src)
        fig_b.layout.title = f"Plot B: source data for {linked_points[i]['sample']}"


def on_hover(trace, points, state):
    if points.point_inds:
        update_bottom(points.point_inds[0])


fig_a.data[0].on_hover(on_hover)
update_bottom(0)

widgets.VBox([fig_a, fig_b])


    'data': [{'mode': 'lines+markers',
              'name': 'derived data',
   …

In [1]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

# Dummy signal
x = np.linspace(0, 20, 400)
y = np.sin(x) + 0.15 * np.random.randn(len(x))

state = {
    "peak_windows": [],
    "baseline_windows": [],
}

styles = {
    "peak": {
        "fill": "rgba(239, 68, 68, 0.18)",
        "line": "#dc2626",
        "text": "#991b1b",
    },
    "baseline": {
        "fill": "rgba(59, 130, 246, 0.16)",
        "line": "#2563eb",
        "text": "#1d4ed8",
    },
}

fig = go.FigureWidget(
    data=[
        go.Scatter(
            x=x,
            y=y,
            mode="lines+markers",
            line=dict(color="#1f77b4", width=2),
            marker=dict(size=6, opacity=0.001),
            name="signal",
        )
    ],
    layout=dict(
        title="Draw x-windows. Choose peak or baseline first. Hold Shift to add more.",
        dragmode="select",
        selectdirection="h",
        hovermode="closest",
        newselection=dict(line=dict(color=styles["peak"]["line"], width=2, dash="dot")),
        activeselection=dict(fillcolor=styles["peak"]["fill"]),
        modebar_remove=["select2d", "lasso2d", "pan2d", "zoom2d"],
    ),
)

mode_toggle = widgets.ToggleButtons(
    options=[("Peak", "peak"), ("Baseline", "baseline")],
    value="peak",
    description="Mode:",
)

drag_toggle = widgets.ToggleButtons(
    options=[("Select", "select"), ("Pan", "pan"), ("Zoom", "zoom")],
    value="select",
    description="Drag:",
)

clear_current_btn = widgets.Button(description="Clear current category")
clear_all_btn = widgets.Button(description="Clear all")
out = widgets.Output()


def merge_windows(windows):
    if not windows:
        return []
    windows = sorted((min(a, b), max(a, b)) for a, b in windows)
    merged = [windows[0]]
    for a, b in windows[1:]:
        last_a, last_b = merged[-1]
        if a <= last_b:
            merged[-1] = (last_a, max(last_b, b))
        else:
            merged.append((a, b))
    return merged


def update_selection_style(kind):
    style = styles[kind]
    with fig.batch_update():
        fig.layout.newselection = dict(line=dict(color=style["line"], width=2, dash="dot"))
        fig.layout.activeselection = dict(fillcolor=style["fill"])


def render_windows():
    shapes = []
    annotations = []

    for kind, key in [("peak", "peak_windows"), ("baseline", "baseline_windows")]:
        style = styles[kind]

        for x0, x1 in state[key]:
            xm = 0.5 * (x0 + x1)
            label = f"{kind}<br>({x0:.2f}, {x1:.2f})"

            shapes.append(
                dict(
                    type="rect",
                    xref="x",
                    yref="paper",
                    x0=x0,
                    x1=x1,
                    y0=0,
                    y1=1,
                    fillcolor=style["fill"],
                    line=dict(color=style["line"], width=2, dash="dot"),
                    layer="below",
                )
            )

            annotations.append(
                dict(
                    x=xm,
                    y=0.98,
                    xref="x",
                    yref="paper",
                    text=label,
                    showarrow=False,
                    xanchor="center",
                    yanchor="top",
                    align="center",
                    bgcolor="rgba(255,255,255,0.90)",
                    bordercolor=style["line"],
                    borderwidth=1,
                    font=dict(size=11, color=style["text"]),
                )
            )

    with fig.batch_update():
        fig.layout.shapes = tuple(shapes)
        fig.layout.annotations = tuple(annotations)
        fig.layout.selections = ()


def show_state():
    with out:
        out.clear_output()
        print("state =", state)


def on_select(trace, points, selector):
    if drag_toggle.value != "select":
        return

    xr = getattr(selector, "xrange", None)
    if xr is None:
        return

    kind = mode_toggle.value
    key = f"{kind}_windows"

    x0, x1 = sorted(map(float, xr))
    state[key].append((x0, x1))
    state[key] = merge_windows(state[key])

    render_windows()
    show_state()


def on_mode_change(change):
    if change["name"] == "value":
        update_selection_style(change["new"])


def on_drag_change(change):
    if change["name"] != "value":
        return

    with fig.batch_update():
        fig.layout.dragmode = change["new"]
        fig.layout.selectdirection = "h"


def clear_current(_):
    key = f"{mode_toggle.value}_windows"
    state[key] = []
    render_windows()
    show_state()


def clear_all(_):
    state["peak_windows"] = []
    state["baseline_windows"] = []
    render_windows()
    show_state()


fig.data[0].on_selection(on_select)
mode_toggle.observe(on_mode_change, names="value")
drag_toggle.observe(on_drag_change, names="value")
clear_current_btn.on_click(clear_current)
clear_all_btn.on_click(clear_all)

update_selection_style(mode_toggle.value)
render_windows()
show_state()

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<b>Instructions:</b> choose <code>Peak</code> or <code>Baseline</code>, "
                "set <code>Drag</code> to <code>Select</code>, then draw horizontal x-windows. "
                "Hold Shift to add more."
            ),
            widgets.HBox([mode_toggle, drag_toggle]),
            fig,
            widgets.HBox([clear_current_btn, clear_all_btn]),
            out,
        ]
    )
)


In [2]:
state


{'peak_windows': [(5.0808368232432075, 9.469551556785527),
  (11.601212998791796, 16.15711686504049)],
 'baseline_windows': []}